# Fundamentos de Ingeniería de Datos
## Transformación: de EXTRACTED a TRANSFORMED

```
INGESTA                      EXTRACCIÓN                          TRANSFORMACIÓN (hoy)
Kaggle → raw_vg_engagement → extracted_raw_vg_engagement    →    diagnosticar · limpiar · tipar · enriquecer
        (SQLite)             (SQLite)                            → df_final (en memoria) + bitácora
```

**Tema 2.2 — Transformaciones:** limpieza, normalización, tipado, *joins* y enriquecimiento.

### ¿Qué es la capa TRANSFORMED?

Son los datos de `extracted_raw_vg_engagement` **limpios, con tipos correctos y listos para analizar**. Es la primera capa en la que **sí cambiamos** los datos.

| Capa | Qué contiene | En la industria (*medallion*) |
|---|---|---|
| RAW | Datos tal como llegaron | Bronze |
| EXTRACTED | Los mismos datos + metadatos de extracción | Bronze |
| **TRANSFORMED** | **Datos limpios, tipados y enriquecidos** | **Silver** |
| SERVING | Modelo para análisis (Unidad 3) | Gold |

> **Regla de oro:** cada capa lee **solo** de la anterior. Este notebook lee `extracted_raw_vg_engagement` (nunca `raw_vg_engagement`) en **modo solo lectura**.

### Configuración del entorno

In [ ]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import PROYECTO_BASE, DB_PATH, inspeccionar_dataset, listar_tablas

print()
print('utils.py importado correctamente')
print(f'Base de datos: {DB_PATH}')
print()

In [ ]:
# ── Librerías, rutas y nombres ─────────────────────────────────────────────

import sqlite3
import pandas as pd
from datetime import datetime




TABLA_EXTRACTED   = 'extracted_raw_vg_engagement'   # origen  (SQLite)
NOMBRE_DESTINO    = 'transformed_raw_vg_engagement' # nombre de la capa limpia (se cargará en la siguiente parte)
VERSION_TRANSFORM = 'v1'                            # súbela si cambias las reglas de limpieza


print()
print(f'Origen : {TABLA_EXTRACTED}  (SQLite, solo lectura)')
print()

---
## PASO 1 — Leer el origen: `extracted_raw_vg_engagement`

Leemos en **solo lectura** y separamos las columnas en dos grupos:

- **Columnas de negocio:** las del dataset (las vamos a limpiar).
- **Metadatos** (empiezan con `_`): **no se tocan**. `_row_hash` nos permitirá rastrear cada fila limpia hasta su origen (*lineage*).

In [ ]:
# ── ¿Qué tablas hay en el .db? ─────────────────────────────────────────────

listar_tablas(DB_PATH);

In [ ]:
# ── Leer extracted_raw_vg_engagement en modo solo lectura ──────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()



COLS_META    = [c for c in df_ext.columns if c.startswith('_')]
COLS_NEGOCIO = [c for c in df_ext.columns if not c.startswith('_')]



print()
print(f'Columnas de negocio : {COLS_NEGOCIO}')
print(f'Metadatos           : {COLS_META}')

inspeccionar_dataset(df_ext, nombre=TABLA_EXTRACTED)

---
## PASO 2 — Diagnóstico de calidad (antes)

**No se limpia lo que no se ha medido.** Antes de transformar, medimos los problemas. Al final repetimos la medición y comparamos: esa comparación va en tu **reporte de validación**.

| Dimensión de calidad | Pregunta | Cómo la medimos |
|---|---|---|
| **Completitud** | ¿Faltan datos? | Nulos por columna |
| **Unicidad** | ¿Hay registros repetidos? | Duplicados exactos y por llave |
| **Validez** | ¿Los valores tienen sentido? | Rangos (años de lanzamiento, rating entre 0 y 5, conteos ≥ 0) |
| **Consistencia** | ¿Los datos cuadran entre sí? | ¿`Playing` ≤ `Plays`? (no puede haber más jugando ahora que los que lo han jugado) |

**Llave de negocio:** el conjunto de columnas que identifica a un registro. En este dataset, un juego se identifica por **título + fecha de lanzamiento** (un *remake* comparte título con el original, pero no la fecha).

In [ ]:
# ── Reglas del dataset (ajústalas a TU dataset) ────────────────────────────

COLS_TEXTO    = ['Title', 'Team', 'Genres']                       # columnas de texto
COLS_CONTEOS  = ['Times Listed', 'Number of Reviews', 'Plays',
                 'Playing', 'Backlogs', 'Wishlist']                # conteos de usuarios (enteros)
COL_FECHA     = 'Release Date'
COL_RATING    = 'Rating'
COL_PLAYS     = 'Plays'                                           # total de usuarios que lo jugaron
COL_PLAYING   = 'Playing'                                         # usuarios jugándolo ahora

ANIO_MIN   = 1980        # primer lanzamiento del dataset
ANIO_MAX   = 2023        # fechas posteriores son juegos aún no lanzados (sin datos reales de uso)

RATING_MIN = 0           # el rating va de 0 a 5
RATING_MAX = 5

In [ ]:
# ── Nulos por columna ──────────────────────────────────────────────────────

nulos = df_ext[COLS_NEGOCIO].isna().sum()

print()
print(f'Total de celdas nulas: {nulos.sum():,}')
print()

nulos

In [ ]:
# ── Registros repetidos ────────────────────────────────────────────────────

# Exactos: todas las columnas del dataset son iguales
dup_exactos = df_ext.duplicated(subset=COLS_NEGOCIO).sum()

print()
print(f'Duplicados exactos: {dup_exactos}')
print()

# Ver los registros repetidos
repetidos = df_ext[df_ext.duplicated(subset=COLS_NEGOCIO, keep=False)]
repetidos[COLS_NEGOCIO]

In [ ]:
# Años fuera de rango (los nulos no cuentan: ya se revisaron aparte)

anio = pd.to_datetime(df_ext[COL_FECHA]).dt.year

anio_invalido = ~anio.between(ANIO_MIN, ANIO_MAX) & anio.notna()

# Filas con al menos un conteo negativo
conteo_negativo = (df_ext[COLS_CONTEOS] < 0).any(axis=1)

# Rating fuera de la escala
rating_invalido = ~df_ext[COL_RATING].between(RATING_MIN, RATING_MAX) & df_ext[COL_RATING].notna()

print()
print(f'Años fuera de [{ANIO_MIN}, {ANIO_MAX}]      : {anio_invalido.sum()}')
print(f'Filas con conteos negativos        : {conteo_negativo.sum()}')
print(f'Rating fuera de [{RATING_MIN}, {RATING_MAX}]           : {rating_invalido.sum()}')
print()

# Ver los registros con año inválido
df_ext.loc[anio_invalido, COLS_NEGOCIO]

In [ ]:
# ── Consistencia: ¿los datos cuadran entre sí? ─────────────────────────────

# Nadie puede estar jugando un juego sin haberlo jugado: Playing debe ser <= Plays
inconsistente = df_ext[COL_PLAYING] > df_ext[COL_PLAYS]

print()
print(f'Filas inconsistentes ({COL_PLAYING} > {COL_PLAYS}) : {inconsistente.sum()}')
print()

# Ver los registros inconsistentes
df_ext.loc[inconsistente, ['Title', COL_FECHA, COL_PLAYS, COL_PLAYING]]

In [ ]:
# ── Resumen del diagnóstico (línea base para comparar después) ─────────────

calidad_antes = pd.Series({
    'filas'                  : len(df_ext),
    'celdas_nulas'           : nulos.sum(),
    'duplicados_exactos'     : dup_exactos,
    'anios_fuera_de_rango'   : anio_invalido.sum(),
    'conteos_negativos'      : conteo_negativo.sum(),
    'rating_fuera_de_rango'  : rating_invalido.sum(),
    'playing_inconsistente'  : inconsistente.sum(),
}, name='antes')

calidad_antes.to_frame()

---
## PASO 3 — Transformaciones

Ahora sí **cambiamos los datos**, atacando lo que encontró el diagnóstico del Paso 2. Aplicamos las reglas **una por una** sobre una copia (`df`) y anotamos cada una en una **bitácora**: qué se hizo y cuántas filas cambió. La bitácora es el *lineage* de las transformaciones y va tal cual en tu reporte.

| # | Transformación | Tipo | ¿Qué problema del Paso 2 resuelve? |
|---|---|---|---|
| 3.1 | Nombres de columnas en `snake_case` | Normalización | Convención de nombres (`Release Date` tiene espacio) |
| 3.2 | Limpiar texto | Normalización | Espacios que separan valores iguales |
| 3.3 | Tratar nulos | Limpieza | **Completitud** |
| 3.4 | Quitar duplicados exactos | Limpieza | **Unicidad** |
| 3.5 | Rangos y consistencia | Limpieza | **Validez** y **consistencia** |
| 3.6 | Convertir tipos de dato | Tipado | Tipos incorrectos (fecha como texto, conteos como decimal) |
| 3.7 | Enriquecer con un catálogo (*join*) | Enriquecimiento | — |
| 3.8 | Columnas derivadas | Enriquecimiento | — |
| 3.9 | Metadatos de transformación | Lineage | — |

In [ ]:
# ── Bitácora de transformaciones ───────────────────────────────────────────

bitacora = []

def registrar(paso, descripcion, filas_antes, filas_despues, afectadas):

    '''Anota un paso en la bitácora y lo muestra'''

    bitacora.append(

        {
        'paso'            : paso,
        'descripcion'     : descripcion,
        'filas_antes'     : filas_antes,
        'filas_despues'   : filas_despues,
        'filas_afectadas' : int(afectadas),
    })


    print(f'[{paso}] {descripcion}')
    print(f'       filas: {filas_antes:,} → {filas_despues:,}   (afectadas: {int(afectadas):,})')



df = df_ext.copy()        # trabajamos sobre una copia: df_ext queda intacto para comparar


print()
print(f'Punto de partida: {len(df):,} filas')
print()

### 3.1 Nombres de columnas en `snake_case`

Una convención única (minúsculas, sin espacios, palabras separadas por `_`) evita errores al escribir SQL: `Release Date`, `release date` y `"Release Date"` dejan de ser un problema.

Como las columnas cambian de nombre, las listas de reglas del Paso 2 también se actualizan con la misma función.

In [ ]:
# ── 3.1 Renombrar columnas ─────────────────────────────────────────────────

def a_snake_case(columnas):

    '''Minúsculas, sin espacios a los lados y con _ en lugar de espacios.'''

    return list(pd.Index(columnas).str.strip().str.lower().str.replace(' ', '_'))


df.columns = a_snake_case(df.columns)

# Las listas de reglas usan ahora los nombres nuevos

COLS_NEGOCIO  = a_snake_case(COLS_NEGOCIO)
COLS_TEXTO    = a_snake_case(COLS_TEXTO)
COLS_CONTEOS  = a_snake_case(COLS_CONTEOS)
COL_FECHA, COL_RATING, COL_PLAYS, COL_PLAYING = a_snake_case([COL_FECHA, COL_RATING, COL_PLAYS, COL_PLAYING])

registrar('3.1', 'Nombres de columnas a snake_case', len(df), len(df), 0)

pd.DataFrame({'antes': df_ext.columns, 'después': df.columns})

### 3.2 Limpiar texto

Espacios al inicio o al final (`'Nintendo '`) o dobles espacios hacen que dos valores iguales se vean distintos al agrupar o al hacer *joins*. `apply()` ejecuta la función de limpieza en cada columna de `COLS_TEXTO`.

In [ ]:
# ── 3.2 Limpiar columnas de texto ──────────────────────────────────────────

def limpiar_texto(columna):

    '''Quita espacios al inicio y al
     final, y deja un solo espacio entre palabras.'''


    return columna.str.strip().str.replace(r'\s+', ' ', regex=True)




texto_antes = df[COLS_TEXTO].copy()

df[COLS_TEXTO] = df[COLS_TEXTO].apply(limpiar_texto)

# Celdas que cambiaron (fillna('') evita contar los nulos como cambios)

cambiadas = (df[COLS_TEXTO].fillna('') != texto_antes.fillna('')).sum().sum()


print()
registrar('3.2', f'Quitar espacios extra en {COLS_TEXTO}', len(df), len(df), cambiadas)
print()


### 3.3 Tratar nulos (completitud)

No hay una regla única: **cada columna se decide según su significado** y la decisión se documenta.

| Columna | Opciones | Decisión | Por qué |
|---|---|---|---|
| `release_date` | eliminar / imputar / dejar nulo | **Eliminar la fila** | Son juegos anunciados que aún no salen (sin fecha confirmada): no tienen datos reales de uso, e inventar una fecha sería un dato falso. Son pocas filas (revisa el Paso 2) |
| resto de columnas | — | Sin cambios | No tienen nulos |

> En tu dataset las decisiones pueden ser otras. Lo importante es **justificarlas** en el reporte.

In [ ]:
# ── 3.3 Nulos ──────────────────────────────────────────────────────────────

# release_date nulo → eliminar fila
antes = len(df)
df = df.dropna(subset=[COL_FECHA])
registrar('3.3', f'Eliminar filas sin {COL_FECHA}', antes, len(df), antes - len(df))

In [ ]:
print(bitacora)

### 3.4 Quitar duplicados exactos (unicidad)

Si todas las columnas del dataset son iguales, la fila está repetida: se conserva la primera y se eliminan las demás. Comparamos solo `COLS_NEGOCIO`, porque los metadatos (`_row_hash`, `_batch_id`) siempre son distintos entre filas.

> Aunque el Paso 2 haya dado 0, la regla se deja en el pipeline: protege las siguientes ejecuciones, cuando lleguen datos nuevos.

In [ ]:
# ── 3.4 Duplicados exactos ─────────────────────────────────────────────────

antes = len(df)
df = df.drop_duplicates(subset=COLS_NEGOCIO, keep='first')

registrar('3.4', 'Eliminar duplicados exactos', antes, len(df), antes - len(df))

### 3.5 Rangos y consistencia (validez y consistencia)

| Regla | Acción si no se cumple |
|---|---|
| Año de `release_date` entre `ANIO_MIN` y `ANIO_MAX` | **Eliminar**: es un juego aún no lanzado |
| Conteos ≥ 0 | **Eliminar**: no existen conteos negativos de usuarios |
| `rating` entre `RATING_MIN` y `RATING_MAX` | **Eliminar**: fuera de la escala |
| `playing` ≤ `plays` | **Marcar** con `playing_consistente = False`, sin eliminar |

¿Por qué marcar y no eliminar la última? La fila puede ser real y el error estar solo en una columna. Eliminarla perdería información; marcarla deja que quien analice decida.

In [ ]:
# ── 3.5 Rangos y consistencia ──────────────────────────────────────────────

# Años fuera de rango → eliminar
print()
antes = len(df)
df = df[pd.to_datetime(df[COL_FECHA]).dt.year.between(ANIO_MIN, ANIO_MAX)]
registrar('3.5a', f'Eliminar {COL_FECHA} fuera de [{ANIO_MIN}, {ANIO_MAX}]', antes, len(df), antes - len(df))




# Conteos negativos → eliminar (se conservan las filas con TODOS los conteos >= 0)
print()
antes = len(df)
df = df[(df[COLS_CONTEOS] >= 0).all(axis=1)]
registrar('3.5b', 'Eliminar filas con conteos negativos', antes, len(df), antes - len(df))



# Rating fuera de escala → eliminar
print()
antes = len(df)
df = df[df[COL_RATING].between(RATING_MIN, RATING_MAX)]
registrar('3.5c', f'Eliminar {COL_RATING} fuera de [{RATING_MIN}, {RATING_MAX}]', antes, len(df), antes - len(df))



# Consistencia → marcar, no eliminar
print()
df['playing_consistente'] = df[COL_PLAYING] <= df[COL_PLAYS]
marcadas = (~df['playing_consistente']).sum()
registrar('3.5d', f'Marcar playing_consistente ({COL_PLAYING} <= {COL_PLAYS})', len(df), len(df), marcadas)
print()

In [ ]:
df.head(5)

### 3.6 Convertir tipos de dato

SQLite guardó los conteos como decimal (`3900.0`) y las fechas como texto. Ahora que no hay nulos, cada columna toma su **tipo correcto**. Parquet guardará estos tipos dentro del archivo.

| Columna | Antes | Después |
|---|---|---|
| `release_date` | texto | **fecha** |
| conteos (`plays`, `playing`, …) | decimal (`float64`) | **entero** (`int64`) |
| `rating` | decimal | decimal redondeado a 1 |
| `_extracted_at` | texto | **fecha y hora** |

https://pandas.pydata.org/docs/reference/arrays.html$0

In [ ]:
# ── 3.6 Tipos de dato ──────────────────────────────────────────────────────

tipos_antes = df.dtypes

df[COL_FECHA]       = pd.to_datetime(df[COL_FECHA]).astype('datetime64[us]')
df[COLS_CONTEOS]    = df[COLS_CONTEOS].astype('int64')
df[COL_RATING]      = df[COL_RATING].round(1)
df['_extracted_at'] = pd.to_datetime(df['_extracted_at']).astype('datetime64[us]')

registrar('3.6', f'Convertir tipos ({COL_FECHA} → fecha; conteos → entero; _extracted_at → fecha)', len(df), len(df), 0)

pd.DataFrame({'antes': tipos_antes, 'después': df.dtypes})

### 3.7 Enriquecer con un catálogo (*join*)

**Enriquecer** es agregar información que el dataset no trae. `genres` tiene varios géneros por juego (`'Adventure, RPG'`); tomamos el primero como **género principal** y lo unimos con su **categoría** usando un catálogo pequeño (una tabla de referencia).

Dos cuidados en todo *join*:

1. **Que no multiplique filas.** `validate='m:1'` hace que pandas marque error si el catálogo tiene un género repetido.
2. **Registros sin pareja.** Con `indicator=True` vemos qué géneros no están en el catálogo; se les asigna `'Otro'`.

In [ ]:
# ── 3.7 Catálogo de géneros → categoría ────────────────────────────────────

# Género principal: el primero de la lista ('Adventure, RPG' → 'Adventure')
df['genero_principal'] = df['genres'].str.split(',').str[0].str.strip()


catalogo_generos = pd.DataFrame([
    ('Shooter', 'Acción'), ('Brawler', 'Acción'), ('Fighting', 'Acción'),
    ('Platform', 'Acción'), ('Arcade', 'Acción'),

    ('Adventure', 'Aventura'), ('Point-and-Click', 'Aventura'), ('Visual Novel', 'Aventura'),

    ('RPG', 'Rol'),

    ('Strategy', 'Estrategia'), ('Turn Based Strategy', 'Estrategia'), ('Real Time Strategy', 'Estrategia'),
    ('Tactical', 'Estrategia'), ('MOBA', 'Estrategia'),

    ('Simulator', 'Simulación'), ('Racing', 'Simulación'), ('Sport', 'Simulación'),

    ('Puzzle', 'Casual'), ('Music', 'Casual'), ('Card & Board Game', 'Casual'),
    ('Quiz/Trivia', 'Casual'), ('Pinball', 'Casual'),

    ('Indie', 'Indie'),

], columns=['genero_principal', 'categoria'])



antes = len(df)

df = df.merge(catalogo_generos, on='genero_principal', how='left', validate='m:1', indicator=True)


# ¿Qué géneros no encontraron categoría?
sin_pareja = df[df['_merge'] == 'left_only']
df['categoria'] = df['categoria'].fillna('Otro')
df = df.drop(columns='_merge')

registrar('3.7', "Join con catálogo de géneros (sin pareja → 'Otro')", antes, len(df), len(sin_pareja))


print()
print(f'Filas sin pareja en el catálogo: {len(sin_pareja)}')
print()

sin_pareja['genero_principal'].value_counts()

In [ ]:
df.head(5)

### 3.8 Columnas derivadas

Columnas calculadas a partir de otras que facilitan el análisis. `anio` y `decada` permitirán agrupar sin repetir el cálculo en cada consulta.

In [ ]:
# ── 3.8 Columnas derivadas ─────────────────────────────────────────────────

df['anio']   = df[COL_FECHA].dt.year               # 2022-02-25 → 2022
df['decada'] = df['anio'] // 10 * 10               # 2022 → 2020

registrar('3.8', 'Columnas derivadas anio y decada', len(df), len(df), 0)

df[['title', COL_FECHA, 'anio', 'decada']].head()

### 3.9 Metadatos de transformación

Igual que en EXTRACTED, agregamos cuándo y con qué versión de reglas se transformó cada fila. Los metadatos de extracción **se conservan**: con `_row_hash` cualquier fila limpia se rastrea hasta `extracted_raw_vg_engagement` y `raw_vg_engagement`.

In [ ]:
# ── 3.9 Metadatos y orden final de columnas ───────────────────────────────

df['_transformed_at']    = pd.Timestamp(datetime.now().replace(microsecond=0))
df['_transformed_at']    = df['_transformed_at'].astype('datetime64[us]')
df['_transform_version'] = VERSION_TRANSFORM



COLUMNAS_FINALES = [

    'title', 'release_date', 'anio', 'decada', 'team', 'genres', 'genero_principal', 'categoria',
    'rating', 'times_listed', 'number_of_reviews', 'plays', 'playing', 'backlogs', 'wishlist', 'playing_consistente',
    '_batch_id', '_extracted_at', '_source_table', '_row_hash', '_transformed_at', '_transform_version',
]


df_final = df[COLUMNAS_FINALES].sort_values(COL_PLAYS, ascending=False).reset_index(drop=True)

registrar('3.9', 'Metadatos de transformación y orden de columnas', len(df_final), len(df_final), 0)



In [ ]:
inspeccionar_dataset(df_final, nombre=f'{NOMBRE_DESTINO} (antes de cargar)')

In [ ]:
# ── La bitácora completa ───────────────────────────────────────────────────

df_bitacora = pd.DataFrame(bitacora)
df_bitacora

---
## PASO 4 — Diagnóstico de calidad (después)

Repetimos **las mismas mediciones** del Paso 2, ahora sobre `df_final`. Las celdas son iguales; solo cambian el DataFrame y los nombres de columnas (ya en minúsculas).

In [ ]:
# ── Nulos ──────────────────────────────────────────────────────────────────

nulos_d = df_final[COLS_NEGOCIO].isna().sum()

print()
print(f'Total de celdas nulas: {nulos_d.sum():,}')
print()

In [ ]:
# ── Duplicados exactos ─────────────────────────────────────────────────────

dup_exactos_d = df_final.duplicated(subset=COLS_NEGOCIO).sum()

print()
print(f'Duplicados exactos: {dup_exactos_d}')
print()

In [ ]:
# ── Validez ────────────────────────────────────────────────────────────────

anio_invalido_d   = ~df_final['anio'].between(ANIO_MIN, ANIO_MAX)
conteo_negativo_d = (df_final[COLS_CONTEOS] < 0).any(axis=1)
rating_invalido_d = ~df_final[COL_RATING].between(RATING_MIN, RATING_MAX)

print()
print(f'Años fuera de [{ANIO_MIN}, {ANIO_MAX}]      : {anio_invalido_d.sum()}')
print(f'Filas con conteos negativos        : {conteo_negativo_d.sum()}')
print(f'Rating fuera de [{RATING_MIN}, {RATING_MAX}]           : {rating_invalido_d.sum()}')
print()

In [ ]:
# ── Consistencia ───────────────────────────────────────────────────────────

inconsistente_d = df_final[COL_PLAYING] > df_final[COL_PLAYS]

print()
print(f'Filas inconsistentes ({COL_PLAYING} > {COL_PLAYS}) : {inconsistente_d.sum()}')
print()

In [ ]:
# ── Antes vs. después ──────────────────────────────────────────────────────

calidad_despues = pd.Series({
    'filas'                  : len(df_final),
    'celdas_nulas'           : nulos_d.sum(),
    'duplicados_exactos'     : dup_exactos_d,
    'anios_fuera_de_rango'   : anio_invalido_d.sum(),
    'conteos_negativos'      : conteo_negativo_d.sum(),
    'rating_fuera_de_rango'  : rating_invalido_d.sum(),
    'playing_inconsistente'  : inconsistente_d.sum(),
}, name='después')

df_calidad = pd.DataFrame({'antes': calidad_antes, 'después': calidad_despues})
df_calidad['cambio'] = df_calidad['después'] - df_calidad['antes']
df_calidad

**Cómo leerla:**
- Nulos, duplicados, años fuera de rango, conteos negativos y rating fuera de rango deben quedar en **0**.
- `playing_inconsistente` puede seguir arriba de 0: esas filas se **marcaron** (`playing_consistente = False`), no se eliminaron.
- La baja de `filas` debe coincidir con la suma de filas eliminadas en la bitácora.

In [ ]:
# ── ¿Cuadra la bitácora con el conteo de filas? ────────────────────────────

eliminadas_bitacora = (df_bitacora['filas_antes'] - df_bitacora['filas_despues']).sum()
eliminadas_real     = len(df_ext) - len(df_final)

print()
print(f'Filas eliminadas según la bitácora : {eliminadas_bitacora:,}')
print(f'Filas eliminadas en realidad       : {eliminadas_real:,}')

if eliminadas_bitacora == eliminadas_real:
    print('Cuadra')
else:
    print('NO CUADRA: revisa la bitácora')
print()

---
## PASO 5 — Carga: escribir la capa TRANSFORMED en Parquet

Hasta aquí, `df_final` **solo vive en memoria**: si cierras el notebook, se pierde. La **carga** (la "L" de ETL) lo guarda en un destino persistente para que otros, y la Unidad 3, lo consuman.

```
extracted_raw_vg_engagement (SQLite)  →  transformaciones  →  transformed_raw_vg_engagement.parquet
```

### ¿Por qué Parquet y no el `.db`?

- En la industria, la capa limpia (*silver*) vive normalmente en archivos **Parquet** dentro de un *data lake*. La base de datos para análisis llega después, al modelar (Unidad 3).
- Parquet **guarda el tipo de cada columna** dentro del archivo: lo que hicimos en 3.6 no se pierde. SQLite y CSV no lo conservan.



### ¿Append o sobrescribir?

| Estrategia | Qué hace | ¿Dónde la usamos? |
|---|---|---|
| **Append** | Agrega filas nuevas al final | EXTRACTED: cada lote es historia que no se puede perder |
| **Sobrescribir** (*full refresh*) | Reemplaza el archivo completo | **TRANSFORMED**: se puede recalcular entero desde EXTRACTED |

Aquí **sobrescribimos**. Si mañana cambia una regla de limpieza, basta con volver a ejecutar el notebook. A esto se le llama **idempotencia**: ejecutar el pipeline una o diez veces deja el mismo resultado.

In [ ]:
# ── Librerías y rutas de la carga ──────────────────────────────────────────

import os
import json

TRANSFORMACION_PATH = f'{PROYECTO_BASE}/Transformacion_de_Datos'
PROCESSED_PATH      = f'{TRANSFORMACION_PATH}/Processed'     # aquí se guardan los Parquet
LOGS_PATH           = f'{TRANSFORMACION_PATH}/Logs'          # log y lineage
REPORTES_PATH       = f'{TRANSFORMACION_PATH}/Reportes'      # evidencias para el reporte


os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(LOGS_PATH, exist_ok=True)
os.makedirs(REPORTES_PATH, exist_ok=True)


PARQUET_DESTINO = f'{PROCESSED_PATH}/{NOMBRE_DESTINO}.parquet'
LOG_PATH        = f'{LOGS_PATH}/transformacion_log.jsonl'
LINEAGE_PATH    = f'{LOGS_PATH}/lineage_{NOMBRE_DESTINO}.json'

print()
print(f'Destino : {PARQUET_DESTINO}')
print()

In [ ]:
# ── Log de operaciones (igual que en EXTRACCIÓN: una línea JSON por operación) ──

def escribir_log(etapa, filas_entrada, filas_salida, estado='exitoso', detalle=None):

    '''Agrega UNA línea al log de transformación.'''


    evento = {

        'timestamp'    : datetime.now().isoformat(timespec='seconds'),
        'etapa'        : etapa,
        'origen'       : TABLA_EXTRACTED,
        'destino'      : PARQUET_DESTINO,
        'filas_entrada': filas_entrada,
        'filas_salida' : filas_salida,
        'version'      : VERSION_TRANSFORM,
        'estado'       : estado,
        'detalle'      : detalle,
    }

    with open(LOG_PATH, 'a', encoding='utf-8') as f:      # 'a' = agregar al final
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')


print()
print('escribir_log() lista')
print()

In [ ]:
# ── Cargar df_final a Parquet ──────────────────────────────────────────────

df_final.to_parquet(PARQUET_DESTINO, index=False)        # si el archivo existe, se sobrescribe

tamanio_mb = os.path.getsize(PARQUET_DESTINO) / 1_048_576

escribir_log('carga_parquet', len(df_ext), len(df_final), detalle=f'{tamanio_mb:.2f} MB')


print()
print(f'{TABLA_EXTRACTED} → {NOMBRE_DESTINO}.parquet')
print(f'  Filas de entrada : {len(df_ext):,}')
print(f'  Filas cargadas   : {len(df_final):,}')
print(f'  Tamaño           : {tamanio_mb:.2f} MB')
print()

### ¿Se conservaron los tipos?

Leemos el archivo de vuelta y comparamos sus tipos con los de `df_final`. Deben ser **iguales**: esa es la ventaja de Parquet.

In [ ]:
# ── Leer el Parquet de vuelta ──────────────────────────────────────────────

df_leido = pd.read_parquet(PARQUET_DESTINO)

print()
print(f'Filas leídas: {len(df_leido):,}')
print()

pd.DataFrame({'en memoria': df_final.dtypes, 'desde Parquet': df_leido.dtypes})

---
## PASO 6 — Lineage y evidencias

El **lineage** documenta el recorrido completo del dato: **origen → transformaciones → destino**. Lo guardamos como JSON junto al log. Cada fila, además, conserva su `_row_hash`, que la vincula con `extracted_raw_vg_engagement` y `raw_vg_engagement`.

También guardamos en `Reportes/` las tablas que van en el **reporte de validación**: la bitácora y la comparación antes vs. después.

In [ ]:
# ── Guardar el lineage ─────────────────────────────────────────────────────

lineage = {

    'recorrido'        : ['Kaggle (CSV)', 'raw_vg_engagement (SQLite)', f'{TABLA_EXTRACTED} (SQLite)', f'{NOMBRE_DESTINO} (Parquet)'],
    'origen'           : TABLA_EXTRACTED,
    'base_de_datos'    : DB_PATH,
    'filas_origen'     : len(df_ext),
    'destino'          : PARQUET_DESTINO,
    'filas_destino'    : len(df_final),
    'estrategia_carga' : 'sobrescribir (full refresh)',
    'version_reglas'   : VERSION_TRANSFORM,
    'generado_en'      : datetime.now().isoformat(timespec='seconds'),
    'trazabilidad'     : f'_row_hash vincula cada fila con {TABLA_EXTRACTED} y raw_vg_engagement',
    'transformaciones' : bitacora,                                    # la bitácora del Paso 3
    'columnas_destino' : df_final.dtypes.astype(str).to_dict(),       # {columna: tipo}
}

with open(LINEAGE_PATH, 'w', encoding='utf-8') as f:
    json.dump(lineage, f, indent=2, ensure_ascii=False, default=int)

print()
print(f'Lineage guardado en: {LINEAGE_PATH}')
print()
print(' → '.join(lineage['recorrido']))
print()

> `default=int` convierte a entero los números de pandas/NumPy (por ejemplo, un `int64`), que el módulo `json` no sabe escribir por sí solo.

In [ ]:
# ── Guardar las evidencias para el reporte ─────────────────────────────────

ruta_bitacora = f'{REPORTES_PATH}/bitacora_{NOMBRE_DESTINO}.csv'
ruta_calidad  = f'{REPORTES_PATH}/calidad_{NOMBRE_DESTINO}.csv'


df_bitacora.to_csv(ruta_bitacora, index=False, encoding='utf-8-sig')       # utf-8-sig: abre bien en Excel
df_calidad.to_csv(ruta_calidad, index_label='metrica', encoding='utf-8-sig')


print()
print('Evidencias guardadas:')
print(f'  {ruta_bitacora}')
print(f'  {ruta_calidad}')
print()



---
## PASO 7 — Validación con DuckDB (sobre el Parquet ya cargado)

Una carga no está terminada hasta que **se valida lo que quedó en el destino**, no lo que había en memoria. Con **DuckDB** consultamos el archivo Parquet directamente con SQL, sin cargarlo a ninguna base.



In [ ]:
# ── DuckDB ─────────────────────────────────────────────────────────────────
# Si no está instalado en el entorno, quita el # de la siguiente línea:
# !pip install duckdb

import duckdb

con = duckdb.connect()        # base en memoria: no crea ningún archivo

P = f"read_parquet('{PARQUET_DESTINO}')"         # así se lee un Parquet dentro de SQL

print()
print(f'DuckDB {duckdb.__version__} listo')
print()

In [ ]:
# ── Primer vistazo: SQL directo sobre el archivo ───────────────────────────

con.sql(f'''
    SELECT title, categoria, anio, rating, plays
    FROM {P}
    ORDER BY plays DESC
    LIMIT 5
''').df()

In [ ]:
# ── Esquema: columnas y tipos guardados en el archivo ──────────────────────

con.sql(f'DESCRIBE SELECT * FROM {P}').df()

In [ ]:
# ── Conteo y nulos en el archivo cargado ───────────────────────────────────

con.sql(f'''
    SELECT COUNT(*)                                              AS filas,
           SUM(CASE WHEN release_date IS NULL THEN 1 ELSE 0 END) AS nulos_release_date,
           SUM(CASE WHEN title        IS NULL THEN 1 ELSE 0 END) AS nulos_title
    FROM {P}
''').df()

In [ ]:
# ── Cerrar la conexión de DuckDB ───────────────────────────────────────────

con.close()

print()
print('Conexión cerrada')
print()